# ERA5 10-Metre Wind Components — Mediterranean Sea
* Use `pystac_client` to search the ERA5-PDS collection on Planetary Computer
* Open the U and V wind Zarr stores directly via `xarray`
* Subset to the Mediterranean domain and compute a monthly mean wind speed
* Visualise with `hvplot` (same approach as the SST notebook)

In [ ]:
import pystac_client
import planetary_computer
import xarray as xr
import numpy as np
import hvplot.xarray

# Connect to the Planetary Computer STAC API
catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)

In [ ]:
# Search for the ERA5 item for a specific month
# U/V wind components are analysis ('an') variables
YEAR_MONTH = "2020-08"

search = catalog.search(collections=["era5-pds"], datetime=YEAR_MONTH)
items = search.item_collection()
print(f"Found {len(items)} items for {YEAR_MONTH}")
for item in items:
    print(f"  {item.id}  (kind={item.properties['era5:kind']})")

In [ ]:
# Select the analysis item — wind components live here
an_item = next(i for i in items if i.properties["era5:kind"] == "an")
signed_item = planetary_computer.sign(an_item)

# Inspect available wind-related assets
wind_assets = [k for k in sorted(signed_item.assets) if "wind" in k]
print("Wind assets:", wind_assets)

In [ ]:
def open_era5_asset(signed_item, asset_key):
    asset = signed_item.assets[asset_key]
    open_kwargs = asset.extra_fields["xarray:open_kwargs"].copy()
    storage_options = open_kwargs.pop("storage_options")
    open_kwargs.pop("engine", None)
    print(f"Opening: {asset.href}")
    return xr.open_zarr(asset.href, storage_options=storage_options, **open_kwargs)

# Actual asset keys in the 'an' item for 10-metre winds
ds_u = open_era5_asset(signed_item, "eastward_wind_at_10_metres")
ds_v = open_era5_asset(signed_item, "northward_wind_at_10_metres")

print(ds_u)
print(ds_v)

In [ ]:
# Mediterranean bounding box
# Latitude:  30 N – 47 N
# Longitude: ERA5 uses 0–360; Med spans -6°→42° E → 354–360 + 0–42
LAT_MIN, LAT_MAX     = 30.0, 47.0
LON_MIN_W, LON_MAX_W = 354.0, 360.0   # -6° to 0°
LON_MIN_E, LON_MAX_E =   0.0,  42.0   #  0° to 42°

def subset_med(da):
    da_lat = da.sel(lat=slice(LAT_MAX, LAT_MIN))
    west   = da_lat.sel(lon=slice(LON_MIN_W, LON_MAX_W))
    east   = da_lat.sel(lon=slice(LON_MIN_E, LON_MAX_E))
    da_med = xr.concat([west, east], dim="lon").sortby("lon")
    # Convert longitude from 0-360 to -180-180
    da_med = da_med.assign_coords(lon=((da_med.lon.values + 180) % 360 - 180))
    return da_med.sortby("lon")

u10_med = subset_med(ds_u["eastward_wind_at_10_metres"])
v10_med = subset_med(ds_v["northward_wind_at_10_metres"])

print("Mediterranean U10 shape:", dict(u10_med.sizes))
print("Mediterranean V10 shape:", dict(v10_med.sizes))

In [ ]:
# Compute monthly mean wind speed: sqrt(u^2 + v^2)
u10_mean = u10_med.mean(dim="time").compute()
v10_mean = v10_med.mean(dim="time").compute()

wind_speed = np.sqrt(u10_mean**2 + v10_mean**2)
wind_speed.attrs["units"] = "m/s"
wind_speed.attrs["long_name"] = "10-Metre Wind Speed"

month_label = np.datetime_as_string(u10_med.time.values[0], unit="M")
print(f"Monthly mean wind speed computed for {month_label}")
print(f"  Min: {float(wind_speed.min()):.2f} m/s   Max: {float(wind_speed.max()):.2f} m/s")

In [ ]:
import holoviews as hv

# Relax tolerance for tiny floating-point jitter in reassigned lon coords
hv.config.image_rtol = 0.01

vmin = float(wind_speed.min())
vmax = float(wind_speed.max())

speed_plot = wind_speed.hvplot(
    x="lon",
    y="lat",
    geo=True,
    rasterize=True,
    cmap="viridis",
    clim=(vmin, vmax),
    clabel="Wind Speed (m/s)",
    alpha=0.85,
    coastline=True,
    tiles="EsriImagery",
    frame_width=800,
    frame_height=450,
    xlim=(-8, 44),
    ylim=(28, 49),
    title=f"ERA5 Mean 10-Metre Wind Speed — Mediterranean ({month_label})",
    fontscale=1.2,
)

speed_plot

In [ ]:
# Also plot U and V components individually side by side
u_plot = u10_mean.hvplot(
    x="lon", y="lat",
    geo=True, rasterize=True,
    cmap="RdBu_r",
    symmetric=True,
    clabel="U10 (m/s)",
    alpha=0.85,
    coastline=True,
    tiles="EsriImagery",
    frame_width=500, frame_height=320,
    xlim=(-8, 44), ylim=(28, 49),
    title=f"ERA5 Mean U10 — Mediterranean ({month_label})",
)

v_plot = v10_mean.hvplot(
    x="lon", y="lat",
    geo=True, rasterize=True,
    cmap="RdBu_r",
    symmetric=True,
    clabel="V10 (m/s)",
    alpha=0.85,
    coastline=True,
    tiles="EsriImagery",
    frame_width=500, frame_height=320,
    xlim=(-8, 44), ylim=(28, 49),
    title=f"ERA5 Mean V10 — Mediterranean ({month_label})",
)

(u_plot + v_plot).cols(1)